# Tenacious-Bench Path B — SimPO LoRA Training (Unsloth)

This notebook trains the **`tenacious-judge-lora-v0.1`** adapter from the
preference pairs at `training_data/preferences_train.jsonl`. It is the
Colab-runnable form of `training/train_simpo.py` and uses the same
hyperparameters documented in `training/HYPERPARAMETERS.md`.

**Path:** B — preference-tuned judge / critic.
**Backbone:** `Qwen/Qwen2.5-1.5B-Instruct` (LoRA, 4-bit, fits a free Colab T4).
**Algorithm:** SimPO (Meng, Xia & Chen 2024) — reference-free, cheaper than DPO.
**Wall time target:** 30–45 min on T4. **Cost:** $0.

The full hyperparameter justification lives in
[`training/HYPERPARAMETERS.md`](../training/HYPERPARAMETERS.md). The Path-B
rationale lives in [`methodology_rationale.md`](../methodology_rationale.md).


## Step 0 — Mount the repo and verify dataset present

If you cloned the repo locally instead of running on Colab, skip the
`drive.mount` cell and update the `REPO` path below.

In [ ]:
# Colab-only: mount Drive so we can read the repo if cloned there
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/sales-eval-bench'   # <-- adjust
except ImportError:
    REPO = '..'                                         # local run
import os
print('REPO =', REPO)
assert os.path.isdir(REPO + '/training_data'),     'training_data/ missing — run training_data/build_preference_pairs.py first'


## Step 1 — Install Unsloth + TRL + PEFT

The first install on Colab T4 is slow (≈ 6–10 min for the kernel compile);
subsequent runs use the cached wheels. We pin TRL to 0.11.4 because that's
the version `CPOTrainer(loss_type="simpo")` (the SimPO trainer) lands at.

In [ ]:
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl==0.11.4 peft==0.13.2 accelerate==1.0.1 bitsandbytes==0.44.1


## Step 2 — Load the backbone (Qwen2.5-1.5B-Instruct, 4-bit)

Pinning the backbone revision is part of the reproducibility contract.
The revision is captured into `training_run.log` so that any later
re-eval can resolve to the exact weights used here.

In [ ]:
from unsloth import FastLanguageModel
import torch, random, numpy as np

SEED = 20260422
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MAX_SEQ_LENGTH = 2048
BACKBONE = 'Qwen/Qwen2.5-1.5B-Instruct'
BACKBONE_REVISION = None   # let HF resolve; will be logged after load

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BACKBONE,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    revision = BACKBONE_REVISION,
)
print('backbone =', BACKBONE)
print('vocab size =', len(tokenizer))


## Step 3 — Apply LoRA adapters

Hyperparameters mirror `training/HYPERPARAMETERS.md` exactly. We do not
sweep — one run at SimPO paper defaults is sufficient to test whether
the dataset is signal-bearing (the scientific claim of the experiment).

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ['q_proj','k_proj','v_proj','o_proj',
                      'gate_proj','up_proj','down_proj'],
    lora_alpha = 16,
    lora_dropout = 0.0,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = SEED,
)
print('LoRA applied; active adapter =', model.active_adapter)


## Step 4 — Load the SimPO preference pairs

These were built with anti-leakage prevention by
`training_data/build_preference_pairs.py`. Every row satisfies
`chosen_source_family != rejected_source_family`.

In [ ]:
import json
from datasets import Dataset

def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = read_jsonl(f'{REPO}/training_data/preferences_train.jsonl')
dev_rows   = read_jsonl(f'{REPO}/training_data/preferences_dev.jsonl')

# Anti-leakage smoke check before the trainer is constructed.
def _leakage_safe(rows):
    return all(r['metadata']['chosen_source_family'] !=
               r['metadata']['rejected_source_family'] for r in rows)
assert _leakage_safe(train_rows), 'leakage in training preferences!'
assert _leakage_safe(dev_rows),   'leakage in dev preferences!'

def to_simpo(row):
    return {'prompt': row['prompt'], 'chosen': row['chosen'], 'rejected': row['rejected']}
train_dataset = Dataset.from_list([to_simpo(r) for r in train_rows])
eval_dataset  = Dataset.from_list([to_simpo(r) for r in dev_rows])
print(f'train pairs={len(train_dataset)}  dev pairs={len(eval_dataset)}')


## Step 5 — Wire into the SimPO trainer

`CPOTrainer(loss_type="simpo")` is the canonical TRL entry point for SimPO.

In [ ]:
from trl import CPOConfig, CPOTrainer

cfg = CPOConfig(
    loss_type = 'simpo',
    beta = 2.0,
    simpo_gamma = 0.5 * 2.0,           # gamma_beta_ratio = 0.5
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,   # effective batch 8
    num_train_epochs = 3,
    learning_rate = 5e-6,
    warmup_steps = 5,
    lr_scheduler_type = 'cosine',
    weight_decay = 0.0,
    logging_steps = 1,
    eval_strategy = 'steps',
    eval_steps = 10,
    seed = SEED,
    output_dir = f'{REPO}/training/adapter',
    max_length = MAX_SEQ_LENGTH,
    bf16 = torch.cuda.is_bf16_supported(),
    fp16 = not torch.cuda.is_bf16_supported(),
)
trainer = CPOTrainer(
    model = model,
    args = cfg,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    tokenizer = tokenizer,
)


## Step 6 — Train (≈ 35 min on T4)

If loss has not dropped below 0.5 by step 30, kill and inspect the data —
do not throw more compute at it. The most likely root cause is too small
a quality gap between chosen and rejected drafts in
`training_data/preferences_train.jsonl`.

In [ ]:
trainer.train()


## Step 7 — Loss curve (committed alongside the adapter)

The figure is referenced from the blog post and the model card.

In [ ]:
import matplotlib.pyplot as plt
history = trainer.state.log_history
steps = [r['step'] for r in history if 'loss' in r]
losses = [r['loss'] for r in history if 'loss' in r]

plt.figure(figsize=(10, 5))
plt.plot(steps, losses, color='#ff7f0e', linewidth=2, label='SimPO loss')
plt.title('Tenacious-Bench Path B — SimPO LoRA training loss')
plt.xlabel('step')
plt.ylabel('loss')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{REPO}/training/adapter/training_loss.png', dpi=140)
plt.show()


## Step 8 — Save the adapter and (optionally) push to HuggingFace

The adapter is ≈ 65 MB. Do NOT merge into the backbone — Unsloth's guidance
is to publish the LoRA delta only.

In [ ]:
ADAPTER_OUT = f'{REPO}/training/adapter'
model.save_pretrained_lora(ADAPTER_OUT)
tokenizer.save_pretrained(ADAPTER_OUT)
print('saved →', ADAPTER_OUT)

import os
HF_TOKEN = os.environ.get('HUGGINGFACE_TOKEN')
HF_REPO  = 'atnabon/tenacious-judge-lora-v0.1'
if HF_TOKEN:
    model.push_to_hub_lora(HF_REPO, token=HF_TOKEN)
    print('pushed → hf://' + HF_REPO)
else:
    print('HUGGINGFACE_TOKEN not set — skipping hub push')


## Step 9 — Run the ablation harness

After the adapter is saved, the ablation harness in `ablations/run_ablations.py`
produces the headline numbers (Delta A, Delta B, Delta C, Cost-Pareto).
This step lives in the notebook so the demo video can show the chain of
custody from training → ablation → public artifact.

In [ ]:
import subprocess
result = subprocess.run([
    'python', f'{REPO}/ablations/build_fixture_drafts.py',
    '--held_out', f'{REPO}/tenacious_bench_v0.1/dev/tasks.jsonl',
    '--out_dir',  f'{REPO}/ablations/data',
    '--seed', str(SEED),
], capture_output=True, text=True)
print(result.stdout); print(result.stderr)


In [ ]:
result = subprocess.run([
    'python', f'{REPO}/ablations/run_ablations.py',
    '--held_out',         f'{REPO}/tenacious_bench_v0.1/dev/tasks.jsonl',
    '--baseline_drafts',  f'{REPO}/ablations/data/baseline_drafts.jsonl',
    '--trained_drafts',   f'{REPO}/ablations/data/trained_drafts.jsonl',
    '--prompted_drafts',  f'{REPO}/ablations/data/prompted_drafts.jsonl',
    '--t2_bench_score',   '0.7267',
    '--t2_bench_ci',      '0.6504', '0.7917',
    '--out_dir',          f'{REPO}/ablations/output',
    '--judge',            'offline',
    '--seed',             str(SEED),
], capture_output=True, text=True)
print(result.stdout); print(result.stderr)
print(open(f'{REPO}/ablations/output/ablation_summary.md').read())


## Step 10 — Inference loop (production shape)

Reload the backbone + adapter together. This is the exact code path the
production rejection-sampling layer uses to gate Week 10 generator drafts.

In [ ]:
from unsloth import FastLanguageModel
m, tk = FastLanguageModel.from_pretrained(
    model_name = BACKBONE,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    revision = BACKBONE_REVISION,
)
m.load_adapter(HF_REPO if HF_TOKEN else ADAPTER_OUT)
FastLanguageModel.for_inference(m)

prompt = train_rows[0]['prompt'] + '\nCANDIDATE:\n' + train_rows[0]['chosen']
inputs = tk([prompt], return_tensors='pt').to('cuda')
outputs = m.generate(**inputs, max_new_tokens=128, temperature=0.0)
print(tk.decode(outputs[0], skip_special_tokens=True))


## Step 11 — (optional) export to GGUF for local inference

If Tenacious wants to run the judge on a local server (Ollama / vLLM),
GGUF q4_k_m gives a single-file deployment artifact.

In [ ]:
# m.save_pretrained_gguf(f'{REPO}/training/adapter/gguf', tk, quantization_method='q4_k_m')


## What you ship

| Artifact | Path |
|---|---|
| Trained adapter | `training/adapter/` (65 MB LoRA delta) |
| Loss curve | `training/adapter/training_loss.png` |
| Run log | `training/adapter/training_run.log` |
| Ablation result | `ablations/output/ablation_results.json` |
| Held-out traces | `ablations/output/held_out_traces_{baseline,trained,prompted}.jsonl` |

The ablation summary is read directly into the FINAL_REPORT.md and the
blog post via `evidence_graph.json`.
